# Прогноз оттока клиентов банка (`Exited`)

В этом ноутбуке решается задача бинарной классификации (`Exited`):
- используются **3 модели**: Logistic Regression, Random Forest, KNN;
- для каждой модели выполняется подбор гиперпараметров;
- как минимум для одной модели используется **`GridSearchCV`** (здесь — для Logistic Regression и KNN).

In [6]:
import json
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

In [7]:
data_path = Path("Churn.csv")
df = pd.read_csv(data_path)

display(df.head())
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


Shape: (10000, 14)

Missing values:
Tenure    909
dtype: int64


In [8]:
drop_cols = ["RowNumber", "CustomerId", "Surname"]
target_col = "Exited"

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical features: ['Geography', 'Gender']


In [9]:
def evaluate_model(model_name, search_obj, X_train, y_train, X_test, y_test):
    search_obj.fit(X_train, y_train)
    best_model = search_obj.best_estimator_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    return {
        "model": model_name,
        "best_params": search_obj.best_params_,
        "cv_best_score_roc_auc": float(search_obj.best_score_),
        "test_accuracy": float(accuracy_score(y_test, y_pred)),
        "test_f1": float(f1_score(y_test, y_pred)),
        "test_roc_auc": float(roc_auc_score(y_test, y_proba)),
        "classification_report": classification_report(y_test, y_pred),
    }

# 1) Logistic Regression + GridSearchCV
# В sklearn>=1.8 не используем penalty в сетке, чтобы избежать FutureWarning.
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, solver="saga")),
    ]
)

lr_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__l1_ratio": [0.0, 1.0],  # 0.0 ~ L2, 1.0 ~ L1
    "model__class_weight": [None, "balanced"],
}

lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
)

# 2) RandomForest + RandomizedSearchCV (ускоренный вариант)
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
    ]
)

rf_param_dist = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced"],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=8,
    scoring="roc_auc",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# 3) KNN + GridSearchCV (ускоренный вариант)
knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier()),
    ]
)

knn_param_grid = {
    "model__n_neighbors": [5, 9, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
)

results = []
results.append(evaluate_model("LogisticRegression (GridSearchCV)", lr_search, X_train, y_train, X_test, y_test))
results.append(evaluate_model("RandomForest (RandomizedSearchCV)", rf_search, X_train, y_train, X_test, y_test))
results.append(evaluate_model("KNeighbors (GridSearchCV)", knn_search, X_train, y_train, X_test, y_test))

c:\Users\Nikita\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


In [10]:
results_sorted = sorted(results, key=lambda d: d["test_roc_auc"], reverse=True)

comparison = pd.DataFrame([
    {
        "model": r["model"],
        "cv_best_score_roc_auc": r["cv_best_score_roc_auc"],
        "test_roc_auc": r["test_roc_auc"],
        "test_f1": r["test_f1"],
        "test_accuracy": r["test_accuracy"],
        "best_params": str(r["best_params"]),
    }
    for r in results_sorted
])

display(comparison)

print("Лучший classification_report:\n")
print(results_sorted[0]["classification_report"])

with open("model_results.json", "w", encoding="utf-8") as f:
    json.dump(results_sorted, f, ensure_ascii=False, indent=2)

print("Результаты сохранены в model_results.json")

,model,cv_best_score_roc_auc,test_roc_auc,test_f1,test_accuracy,best_params
0,RandomForest (RandomizedSearchCV),0.859680,0.862920,0.623126,0.8240,"{'model__n_estimators': 400, 'model__min_sampl..."
1,KNeighbors (GridSearchCV),0.819479,0.824260,0.456446,0.8440,"{'model__n_neighbors': 15, 'model__p': 2, 'mod..."
2,LogisticRegression (GridSearchCV),0.766493,0.777369,0.500436,0.7135,"{'model__C': 0.1, 'model__class_weight': 'bala..."


Лучший classification_report:

              precision    recall  f1-score   support

           0       0.92      0.85      0.89      1593
           1       0.55      0.71      0.62       407

    accuracy                           0.82      2000
   macro avg       0.74      0.78      0.75      2000
weighted avg       0.85      0.82      0.83      2000

Результаты сохранены в model_results.json
